# Φάση Δ: Advanced Technique

Εμείς επιλέξαμε το K-Means


In [7]:
# Φάση Δ: Advanced Technique 
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.fpm import FPGrowth
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import PCA

spark = SparkSession.builder.appName("Advanced_Techniques").master("local[*]").getOrCreate()

# --- 1. ASSOCIATION RULES ---
print("Εξαγωγή Κανόνων Συσχέτισης...")
df_silver = spark.read.parquet("../data/train_silver.parquet")

transactions_df = df_silver.withColumn(
    "items",
    F.array_remove(F.array(
        F.when(F.col("stroke") == 1, "Stroke").otherwise(""),
        F.when(F.col("hypertension") == 1, "Hypertension").otherwise(""),
        F.when(F.col("heart_disease") == 1, "Heart_Disease").otherwise(""),
        F.when(F.col("age") > 65, "Age_>_65").otherwise("")
    ), "")
).filter(F.size("items") > 0).select("items")

fpGrowth = FPGrowth(itemsCol="items", minSupport=0.01, minConfidence=0.1) 
rules = fpGrowth.fit(transactions_df).associationRules

print("Κανόνες Κινδύνου για Εγκεφαλικό:")
rules.filter(F.array_contains(F.col("consequent"), "Stroke")).sort(F.col("lift").desc()).show(truncate=False)

# --- 2. CLUSTERING (K-MEANS) ---
print("Εκτέλεση K-Means...")
train_gold = spark.read.parquet("../data/train_gold.parquet")
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=3, seed=42)
kmeans_preds = kmeans.fit(train_gold).transform(train_gold)

pca = PCA(k=2, inputCol="features", outputCol="pca_features")
kmeans_pca = pca.fit(kmeans_preds).transform(kmeans_preds)

kmeans_pca.select("stroke", "cluster", "pca_features").write.mode("overwrite").parquet("../data/preds_kmeans.parquet")
spark.stop()

Εξαγωγή Κανόνων Συσχέτισης...
Κανόνες Κινδύνου για Εγκεφαλικό:
+-------------------------+----------+-------------------+------------------+--------------------+
|antecedent               |consequent|confidence         |lift              |support             |
+-------------------------+----------+-------------------+------------------+--------------------+
|[Hypertension, Age_>_65] |[Stroke]  |0.23076923076923078|1.2396878483835005|0.03776978417266187 |
|[Heart_Disease, Age_>_65]|[Stroke]  |0.21014492753623187|1.128894489953091 |0.026079136690647483|
|[Heart_Disease]          |[Stroke]  |0.18018018018018017|0.9679244461853157|0.03597122302158273 |
|[Age_>_65]               |[Stroke]  |0.1671018276762402 |0.8976677892559377|0.11510791366906475 |
|[Hypertension]           |[Stroke]  |0.14039408866995073|0.7541943314057257|0.051258992805755396|
+-------------------------+----------+-------------------+------------------+--------------------+

Εκτέλεση K-Means...
